In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Conv2D,Activation,Dropout,Flatten,MaxPooling2D
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import confusion_matrix

I0000 00:00:1794463896.321844    8672 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794463896.499111    8672 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794463898.114957    8672 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
dataset_path = "/home/lokesh/Loki/Projects/GEN AI/Face mask detection/dataset"

In [3]:
X = []
y = []
categories = ["with_mask", "without_mask"]
for label, category in enumerate(categories):
    folder_path = os.path.join(dataset_path, category)
    for image_name in os.listdir(folder_path):
        image_path = os.path.join(folder_path, image_name)
        try:
            image = keras.utils.load_img( image_path, target_size=(128,128) )
            image = keras.utils.img_to_array(image)
            X.append(image)
            y.append(label)
        except Exception as e:
            print("Skipping:", image_name)

/home/lokesh/Loki/Projects/GEN AI/.venv/lib/python3.14/site-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [4]:
X = np.array(X)
y = np.array(y)

In [5]:
print(X.shape)
print(y.shape)

(3846, 128, 128, 3)
(3846,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y )

In [7]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [8]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(3076, 128, 128, 3)
(770, 128, 128, 3)
(3076,)
(770,)


In [9]:
model = Sequential()
model.add(Conv2D(28,kernel_size=(3,3),input_shape=(128,128,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

/home/lokesh/Loki/Projects/GEN AI/.venv/lib/python3.14/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1794463909.637363    8672 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50

W0000 00:00:1794463909.848067    8672 cpu_allocator_impl.cc:174] Allocation of 29491200 exceeds 10% of free system memory.
W0000 00:00:1794463909.870800    8672 cpu_allocator_impl.cc:174] Allocation of 29491200 exceeds 10% of free system memory.
W0000 00:00:1794463909.886303    8672 cpu_allocator_impl.cc:174] Allocation of 29491200 exceeds 10% of free system memory.


In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=15,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/15


W0000 00:00:1794463910.917582    8672 cpu_allocator_impl.cc:174] Allocation of 604766208 exceeds 10% of free system memory.
W0000 00:00:1794463912.449268    8672 cpu_allocator_impl.cc:174] Allocation of 29491200 exceeds 10% of free system memory.


97/97 ━━━━━━━━━━━━━━━━━━━━ 33s 326ms/step - accuracy: 0.8140 - loss: 0.4967 - val_accuracy: 0.8948 - val_loss: 0.2487
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 33s 341ms/step - accuracy: 0.9262 - loss: 0.2076 - val_accuracy: 0.9377 - val_loss: 0.1589
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 31s 323ms/step - accuracy: 0.9477 - loss: 0.1446 - val_accuracy: 0.9584 - val_loss: 0.1213
Epoch 4/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 30s 311ms/step - accuracy: 0.9633 - loss: 0.1145 - val_accuracy: 0.9584 - val_loss: 0.1184
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 31s 319ms/step - accuracy: 0.9714 - loss: 0.0779 - val_accuracy: 0.9675 - val_loss: 0.1118
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.9759 - loss: 0.0671

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.save("face_mask_cnn.keras")

In [ ]:
import cv2
import numpy as np
from tensorflow import keras

In [ ]:
model = keras.models.load_model("face_mask_cnn.keras")

In [ ]:
print(cv2.__file__)
print(cv2.__version__)
print(hasattr(cv2, "CascadeClassifier"))

In [ ]:
print([name for name in dir(cv2) if "Cascade" in name])

In [ ]:
print(hasattr(cv2, "objdetect"))

In [ ]:
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [ ]:
print("Face detector loaded:", not face_detector.empty())

In [ ]:
cap = cv2.VideoCapture(0)

print("Camera available:", cap.isOpened())

cap.release()

In [ ]:

import cv2
import numpy as np
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open webcam")
else:
    print("Webcam started!")
    print("Press Q to quit.")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Cannot read webcam frame")
                break
            gray = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2GRAY
            )
            faces = face_detector.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(30, 30)
            )
            for (x, y, w, h) in faces:
                face = frame[y:y+h, x:x+w]
                face = cv2.cvtColor(
                    face,
                    cv2.COLOR_BGR2RGB
                )
                face = cv2.resize(
                    face,
                    (128, 128)
                )
                face = face.astype("float32") / 255.0
                face = np.expand_dims(face, axis=0)
                prediction = model.predict(
                    face,
                    verbose=0
                )[0][0]

                if prediction >= 0.5:
                    label = "NO MASK"
                    confidence = prediction * 100
                    color = (0, 0, 255)

                else:
                    label = "MASK"
                    confidence = (1 - prediction) * 100
                    color = (0, 255, 0)

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x+w, y+h),
                    color,
                    2
                )
                text = f"{label}: {confidence:.1f}%"
                cv2.putText(
                    frame,
                    text,
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2
                )

            cv2.imshow(
                "Face Mask Detection",
                frame
            )
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

        print("Webcam stopped.")
